<tabla align="centro">
  <td align="center"><a target="_blank" href="http://introtodeeplearning.com">
        <img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
      Visite el aprendizaje profundo del MIT</a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com/github/MITDeepLearning/introtodeeplearning/blob/master/lab3/solutions/LLM_Finetuning_Solution.ipynb">
        <img src="https://i.ibb.co/2P3SLwK/colab.png" style="padding-bottom:5px;" />Ejecutar en Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/MITDeepLearning/introtodeeplearning/blob/master/lab3/solutions/LLM_Finetuning_Solution.ipynb">
        <img src="https://i.ibb.co/xfJbPmL/github.png" height="70px" style="padding-bottom:5px;"  />Ver código fuente en GitHub</a></td>
</tabla>

# Información de derechos de autor

In [ ]:
# Copyright 2026 MIT Introducción al aprendizaje profundo. Reservados todos los derechos.
# 
# Licenciado bajo la Licencia MIT. No puede utilizar este archivo excepto en cumplimiento
# con la Licencia. Uso y/o modificación de este código fuera del MIT Introducción
# al Deep Learning debe hacer referencia a:
# 
# © MIT Introducción al aprendizaje profundo
# http://intotodeeplearning.com
# 

# Laboratorio 3: Ajuste del modelo de lenguaje grande (LLM)

En esta práctica de laboratorio, ajustará un modelo de lenguaje grande (LLM) de miles de millones de parámetros. Analizaremos varios conceptos fundamentales de los LLM, incluida la tokenización, las plantillas y el ajuste. Esta práctica de laboratorio proporciona un proceso completo para ajustar un modelo de lenguaje para generar respuestas en un estilo específico, y explorará no solo el ajuste del modelo de lenguaje, sino también formas de evaluar el rendimiento de un modelo de lenguaje.

Utilizará [Liquid AI's](https://www.liquid.ai/) [LFM2-1.2B](https://huggingface.co/LiquidAI/LFM2-1.2B) como modelo de lenguaje base para realizar ajustes; El modelo [Gemini 2.5](https://blog.google/technology/google-deepmind/gemini-model-thinking-updates-march-2025/) de Google como modelo de "juez" de evaluación; y [Opik](https://www.comet.com/site/products/opik/) de Comet ML como marco para una evaluación LLM optimizada.

Primero, descarguemos el paquete de aprendizaje profundo del MIT, instalemos las dependencias e importemos los paquetes relevantes que necesitaremos para esta práctica de laboratorio.

In [ ]:
# Instalar e importar utilidades de aprendizaje profundo del MIT
!pip install mitdeeplearning > /dev/null 2>&1
import mitdeeplearning as mdl

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from lion_pytorch import Lion

# Parte 1: perfeccionar un LLM según el estilo

En la primera parte de esta práctica de laboratorio, ajustaremos un LLM como un chatbot que puede generar respuestas en un estilo específico. Usaremos [Liquid AI LFM2-1.2B model](https://huggingface.co/LiquidAI/LFM2-1.2B) como modelo de lenguaje base para realizar ajustes.

## 1.1: Plantillas y tokenización

### 1.1.1: Plantillas

Los modelos de lenguaje que funcionan como chatbots pueden generar respuestas a las consultas de los usuarios, pero ¿cómo lo hacen? Necesitamos brindarles una manera de comprender la conversación y generar respuestas de manera coherente: alguna estructura de lo que son entradas y salidas.

[Templating](https://huggingface.co/docs/transformers/main/chat_templating) es una forma de formatear entradas y salidas en una estructura consistente que un modelo de lenguaje pueda entender. Implica agregar fichas o marcadores especiales para indicar diferentes partes de la conversación, como quién está hablando y dónde comienzan y terminan los turnos. Esta estructura ayuda al modelo a aprender el formato adecuado para generar respuestas y mantener un flujo de conversación coherente. Sin plantillas, es posible que el modelo no sepa cómo formatear adecuadamente sus resultados o distinguir entre diferentes hablantes en una conversación.

Comencemos definiendo algunas plantillas básicas para el chatbot basado en LFM2, para turnos donde el usuario hace una pregunta y el modelo responde con una respuesta.

In [ ]:
# Plantilla básica de preguntas y respuestas
template_without_answer = "<|startoftext|><|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
template_with_answer = template_without_answer + "{answer}<|im_end|>\n"

# Intentemos poner algo en la plantilla para ver cómo queda.
print(template_with_answer.format(question="What is your name?", answer="My name is Lili!"))

### 1.1.2: Tokenización

Para operar sobre el lenguaje, necesitamos preparar el texto para el modelo. Fundamentalmente podemos pensar en el lenguaje como una secuencia de "fragmentos" de texto. Podemos dividir el texto en fragmentos individuales y luego asignar estos fragmentos a tokens numéricos; en conjunto, este es el proceso de [tokenization](https://huggingface.co/docs/transformers/main/tokenizer_summary). Luego, los tokens numéricos se pueden introducir en un modelo de lenguaje.

Existen varios enfoques comunes para tokenizar texto en lenguaje natural:

1. **Tokenización basada en palabras**: divide el texto en palabras individuales. Si bien es simple, esto puede generar vocabularios extensos y no maneja bien palabras desconocidas.

2. **Tokenización basada en caracteres**: divide el texto en caracteres individuales. Si bien esto implica un vocabulario muy pequeño, produce secuencias largas y pierde significado a nivel de palabra.

3. **Tokenización de subpalabras**: divide las palabras en unidades más pequeñas (subpalabras) según su frecuencia. El enfoque más popular y comúnmente utilizado es [byte-pair encoding (BPE)](https://en.wikipedia.org/wiki/Byte_pair_encoding), que fusiona de forma iterativa los pares de caracteres más frecuentes. Los modelos de lenguaje modernos suelen utilizar la tokenización de subpalabras, ya que equilibra el tamaño del vocabulario y la longitud de la secuencia y, al mismo tiempo, manejan palabras desconocidas de forma eficaz dividiéndolas en unidades de subpalabras conocidas.

En esta práctica usaremos el tokenizador del modelo LFM2, que usa BPE. Carguémoslo e inspeccionémoslo.

In [ ]:
# Cargue el tokenizador para Liquid AI LFM2-1.2B
model_id = "LiquidAI/LFM2-1.2B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# ¿Qué tamaño tiene el tokenizador?
print(f"Vocab size: {len(tokenizer.get_vocab())}")

No solo necesitamos poder tokenizar el texto en tokens (codificar), sino también destokenizar los tokens nuevamente en texto (decodificar). Nuestro tokenizador tendrá:
1. una función `codificar` para tokenizar el texto en tokens, y
2. una función de "decodificación" para volver a convertir el token en texto para que podamos leer las salidas del modelo.

Probemos ambos pasos e inspeccionemos para comprender mejor cómo funciona.

In [ ]:
# Probemos ambos pasos:
text = "Here is some sample text!"
print(f"Original text: {text}")

# Tokenizar el texto
tokens = tokenizer.encode(text, return_tensors="pt")
print(f"Encoded tokens: {tokens}")

# Decodificar las fichas
decoded_text = tokenizer.decode(tokens[0], skip_special_tokens=True)
print(f"Decoded text: {decoded_text}")

Esto es realmente genial. Ahora tenemos una forma de entrar y salir del espacio simbólico.

Para "chatear" con nuestro chatbot LLM, necesitamos usar el tokenizador y la plantilla de chat juntos para que el modelo responda a la pregunta del usuario. Podemos usar las plantillas definidas anteriormente para construir una pregunta para el modelo, sin la respuesta.

In [ ]:
prompt = template_without_answer.format(question="What is the capital of France? Use one word.")
print(prompt)

Si alimentáramos esto al modelo, vería que ahora es el comienzo del turno del modelo y generaría la respuesta a esta pregunta.

## 1.2: Comenzando con el LLM

Ahora que tenemos una manera de preparar nuestros datos, ¡estamos listos para trabajar con nuestro LLM!

Los LLM como LFM2 están entrenados en un gran corpus de texto, en la tarea de predecir el siguiente token en una secuencia, dados los tokens anteriores. A esta tarea de entrenamiento la llamamos "próxima predicción del token"; También puede verlo llamado "modelado de lenguaje causal" o "modelado de lenguaje autorregresivo". Podemos aprovechar los modelos entrenados de esta manera para generar texto nuevo tomando muestras de la distribución de probabilidad predicha sobre el siguiente token.

Carguemos el modelo LFM2 y comencemos a trabajar con él. Construiremos un mensaje en forma de plantilla de chat y lo tokenizaremos. Luego, lo introduciremos en el modelo para predecir las probabilidades del próximo token. Finalmente, obtendremos el siguiente token (que sigue siendo numérico) y lo decodificaremos en texto.

In [ ]:
# Cargue el modelo. Tenga en cuenta que esto puede tardar unos minutos.
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

In [ ]:
# ## Juntándolo para solicitar al modelo y generar una respuesta ###

# 1. Construya el mensaje en forma de plantilla de chat.
question = "What is the capital of France? Use one word."
prompt = template_without_answer.format('''TODO''') # TODO

# 2. Tokenizar el mensaje
tokens = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

# 3. Analice el modelo para predecir las siguientes probabilidades simbólicas.
with torch.no_grad():
    output = '''TODO''' # TODO

    probs = F.softmax(output.logits, dim=-1)

# 4. Consigue el siguiente token, según la máxima probabilidad.
next_token = torch.argmax(probs[0, -1, :]).item()

# 5. Decodifica el siguiente token
next_token_text = '''TODO''' # TODO

print(f"Prompt: {prompt}")
print(f"Predicted next token: {next_token_text}")

Tenga en cuenta que el modelo no puede predecir la respuesta a la pregunta, ¡solo puede predecir el siguiente token de la secuencia! Para preguntas más complejas, no podemos generar simplemente un token, sino que necesitamos generar una secuencia de tokens.

Esto se puede hacer realizando el proceso anterior de forma iterativa, paso a paso: después de cada paso, reintroducimos el token generado en el modelo y predecimos el siguiente token nuevamente.

En lugar de hacer esto manualmente nosotros mismos, podemos usar la funcionalidad [`model.generate()`](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation#transformers.GenerationMixin.generate) incorporada del modelo (compatible con la biblioteca Transformers de HuggingFace) para generar una cantidad de tokens `max_new_tokens` y decodificar la salida nuevamente en texto.

In [ ]:
prompt = template_without_answer.format(question="What does MIT stand for?")
tokens = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
output = model.generate(tokens, max_new_tokens=20)
print(tokenizer.decode(output[0]))

¡Ahora tenemos el proceso básico para generar texto con un LLM!

## 1.3: Ajuste fino

El ajuste fino es una técnica que nos permite adaptar una red neuronal previamente entrenada para que se adapte mejor a una tarea, dominio o estilo posterior, entrenando más el modelo con nuevos datos. Al entrenar más el modelo en un conjunto de datos cuidadosamente seleccionado, podemos modificar su comportamiento, estilo o capacidades. El ajuste fino se utiliza en una variedad de aplicaciones, no solo en el modelado de lenguaje. Pero en el modelado del lenguaje, el ajuste fino se puede utilizar para:
- Adaptar el estilo de escritura del modelo.
- Mejorar el rendimiento en tareas o dominios específicos
- Enseñar al modelo nuevas capacidades o conocimientos.
- Reducir comportamientos o prejuicios no deseados.

En esta práctica de laboratorio, ajustará el LFM2 LLM para adaptar el estilo de escritura del modelo. Recuerde que en el Laboratorio 1 desarrolló un modelo de secuencia basado en RNN para generar canciones populares irlandesas. Continuando con nuestro tema irlandés, primero ajustaremos el LLM para chatear al estilo de un duende.

![Let's Dance!](http://33.media.tumblr.com/3d223954ad0a77f4e98a7b87136aa395/tumblr_nlct5lFVbF1qhu7oio1_500.gif)

Hemos preparado un conjunto de datos de preguntas y respuestas donde las preguntas están en estilo inglés estándar (es decir, estilo "base") y las respuestas están en estilo "leprechaun" (escritas por otro LLM). Carguemos el conjunto de datos e inspeccionémoslo.

In [ ]:
train_loader, test_loader = mdl.lab3.create_dataloader(style="leprechaun")

sample = train_loader.dataset[44]
question = sample['instruction']
answer = sample['response']
answer_style = sample['response_style']

print(f"Question: {question}\n\n" +
      f"Original Answer: {answer}\n\n" +
      f"Answer Style: {answer_style}")

### 1.3.1: Función de chat

Antes de comenzar con el ajuste, crearemos una función para chatear fácilmente con el modelo, tanto para que podamos monitorear su progreso durante el transcurso del ajuste como para generar respuestas a preguntas.

Recuerde nuestros pasos principales de antes:
1. Construya la pregunta utilizando la plantilla.
2. Tokenizar el texto
3. Introduzca los tokens a través del modelo para predecir las probabilidades del próximo token.
4. Decodificar los tokens predichos en texto.

Utilice estos pasos para crear la función de "chat" a continuación.

In [ ]:
def chat(question, max_new_tokens=32, temperature=0.7, only_answer=False):
    # 1. Construya el mensaje usando la plantilla.
    prompt = template_without_answer.format('''TODO''') # TODO

    # 2. Tokenizar el texto
    input_ids = tokenizer('''TODO''', '''TODO''').to(model.device) # TODO

    # 3. Analice el modelo para predecir las siguientes probabilidades simbólicas.
    with torch.no_grad():
        outputs = model.generate('''TODO''', do_sample=True, max_new_tokens=max_new_tokens, temperature=temperature) # TODO

    # 4. Solo devuelve la respuesta si only_answer es Verdadero
    output_tokens = outputs[0]
    if only_answer:
        output_tokens = output_tokens[input_ids['input_ids'].shape[1]:]

    # 5. Decodifica las fichas
    result = tokenizer.decode('''TODO''', skip_special_tokens=True) # TODO

    return result


¡Intentemos chatear con el modelo ahora para probar si funciona! Tenemos una pregunta de muestra aquí (continuando con el tema irlandés); ¡No dudes en probar otras preguntas!

In [ ]:
# ¡Intentemos chatear con el modelo ahora para probar si funciona!
answer = chat(
    "What is the capital of Ireland?",
    only_answer=True,
    max_new_tokens=32,
)

print(answer)

# ## TODO: Experimente haciéndole al modelo diferentes preguntas y valores de temperatura, ¡y vea cómo responde!

### 1.3.2: Ajuste fino eficiente en los parámetros

Durante el ajuste fino, los pesos del modelo se actualizan para adaptarse mejor al conjunto de datos y/o tarea de ajuste fino. Actualizar todos los pesos en un modelo de lenguaje como LFM2-1.2B, que tiene aproximadamente mil millones de parámetros, es computacionalmente costoso. Existen muchas técnicas para hacer que el ajuste fino sea más eficiente.

Usaremos una técnica llamada [LoRA](https://arxiv.org/abs/2106.09685) (adaptación de rango bajo) para hacer que el proceso de ajuste sea más eficiente. LoRA es una forma de ajustar los LLM de manera muy eficiente actualizando solo un pequeño subconjunto de los parámetros del modelo, y funciona agregando matrices de bajo rango entrenables al modelo. Si bien no entraremos en detalles sobre LoRA aquí, puedes leer más al respecto en [LoRA paper](https://arxiv.org/abs/2106.09685). Usaremos la biblioteca [`peft`](https://pypi.org/project/peft/) para aplicar LoRA al modelo LFM.

In [ ]:
# LoRA es una forma de ajustar los LLM de manera muy eficiente actualizando solo un pequeño subconjunto de los parámetros del modelo.

def apply_lora(model):
    # Definir la configuración de LoRA
    lora_config = LoraConfig(
        r=8, # rango de las matrices LoRA
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"
        ],
    )

    # Aplicar LoRA al modelo.
    lora_model = get_peft_model(model, lora_config)
    return lora_model

model = apply_lora(model)

# Imprima la cantidad de parámetros entrenables después de aplicar LoRA
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"number of trainable parameters: {trainable_params}")
print(f"total parameters: {total_params}")
print(f"percentage of trainable parameters: {trainable_params / total_params * 100:.2f}%")

### 1.3.3: Cálculo de pases y pérdidas

Ahora definamos una función para realizar un paso directo a través del LLM y calcular la pérdida. El pase directo nos proporciona los logits (que reflejan la distribución de probabilidad sobre el siguiente token) para el siguiente token. Podemos calcular la pérdida comparando los logits predichos con el siguiente token verdadero: nuestra etiqueta objetivo. Tenga en cuenta que esto es efectivamente un problema de clasificación. Entonces, nuestra pérdida puede ser capturada por la pérdida de entropía cruzada y podemos usar la función [`nn.functional.cross_entropy`](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) de PyTorch para calcularla.

In [ ]:
def forward_and_compute_loss(model, tokens, mask, context_length=512):
    # Truncar a la longitud del contexto
    tokens = tokens[:, :context_length]
    mask = mask[:, :context_length]

    # Construir la entrada, salida y máscara.
    x = tokens[:, :-1]
    y = tokens[:, 1:]
    mask = mask[:, 1:]

    # Pase directo para calcular logits
    logits = model(x).logits

    # Pérdida de cálculo
    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        y.view(-1),
        reduction="none"
    )

    # Ocultar la pérdida por tokens sin respuesta
    loss = loss[mask.view(-1)].mean()

    return loss

### 1.3.4: Bucle de entrenamiento para ajuste fino

Con esta función para calcular la pérdida, ahora podemos definir un bucle de entrenamiento para ajustar el modelo usando LoRA. Este ciclo de capacitación tiene los mismos componentes principales que hemos visto antes en otros laboratorios:
1. Obtenga un lote de datos del conjunto de datos (usando DataLoader)
2. Introduzca los datos a través del modelo para completar un pase hacia adelante y calcular la pérdida.
3. Pase hacia atrás para actualizar los pesos del modelo.

Los datos de nuestro DataLoader son inicialmente texto y no están estructurados en nuestra plantilla de preguntas y respuestas. Entonces, en el paso (1), necesitaremos formatear los datos en nuestra plantilla de preguntas y respuestas previamente definida y luego tokenizar el texto.

Nos preocupamos por la respuesta del modelo a la pregunta; los tokens de "respuesta" son la parte del texto que queremos predecir y calcular la pérdida. Entonces, después de tokenizar el texto, debemos indicarle al modelo qué tokens son parte de la "respuesta" y cuáles son parte de la "pregunta". Podemos hacer esto calculando una máscara para los tokens de respuesta y luego usando esta máscara para calcular la pérdida.

Finalmente, completaremos el paso hacia atrás para actualizar los pesos del modelo.

Juntemos todo esto en el ciclo de entrenamiento a continuación.

In [ ]:
# ## Bucle de entrenamiento ###

def train(model, dataloader, tokenizer, max_steps=200, context_length=512, learning_rate=1e-4):
    losses = []

    # Aplicar LoRA al modelo.
    model = '''TODO''' # TODO

    optimizer = Lion(model.parameters(), lr=learning_rate)

    # Bucle de entrenamiento
    for step, batch in enumerate(dataloader):
        question = batch["instruction"][0]
        answer = batch["response_style"][0]

        # Formatee la pregunta y la respuesta en la plantilla.
        text = template_with_answer.format('''TODO''', '''TODO''') # TODO

        # Tokenice el texto y calcule la máscara para la respuesta.
        ids = tokenizer(text, return_tensors="pt", return_offsets_mapping=True).to(model.device)
        mask = ids["offset_mapping"][:,:,0] >= text.index(answer)

        # Introduzca los tokens a través del modelo y calcule la pérdida.
        loss = forward_and_compute_loss('''TODO''') # TODO

        # pase hacia atrás
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        # monitorear el progreso
        if step % 10 == 0:
            print(chat("What is the capital of France?", only_answer=True))
            print(f"step {step} loss: {torch.mean(torch.tensor(losses)).item()}")
            losses = []

        if step > 0 and step % max_steps == 0:
            break

    return model


In [ ]:
# ¡Llame a la función de tren para ajustar el modelo! Sugerencia: comenzará a ver resultados después de al menos 100 pasos.
model = train('''TODO''') # TODO

¡Intentemos charlar con el modelo nuevamente para ver cómo ha cambiado!

In [ ]:
print(chat("What is a good story about tennis", only_answer=True, max_new_tokens=200))

# Parte 2: Evaluación de un LLM adaptado al estilo

¿Cómo sabemos si el modelo está funcionando bien? ¿En qué medida el estilo del modelo coincide con el estilo de un duende? Como puede ver en el ejemplo anterior, determinar si una respuesta generada es buena o no puede parecer cualitativo y puede resultar difícil medir qué tan bien está funcionando el modelo.

Si bien se han desarrollado puntos de referencia para evaluar el rendimiento de los modelos de lenguaje en una variedad de tareas, estos puntos de referencia no siempre son representativos del rendimiento del modelo en el mundo real. Por ejemplo, un modelo puede funcionar bien en un punto de referencia pero mal en una tarea más realista. Los puntos de referencia también están limitados en el alcance de las tareas que pueden cubrir y las capacidades que pueden reflejar, y puede haber dudas sobre si los datos del punto de referencia se utilizaron para entrenar el modelo. La generación de datos sintéticos y las tareas sintéticas son una forma de abordar estas limitaciones, y esta es un área activa de investigación.

También podemos convertir una evaluación cualitativa de una respuesta generada en cuantitativa enviando a alguien o algo para "juzgar" los resultados. En esta práctica de laboratorio, usaremos una técnica llamada [LLM as a judge](https://arxiv.org/abs/2306.05685) para hacer exactamente esto. Esto implica utilizar un LLM más grande para calificar los resultados de un LLM más pequeño. El LLM más grande se utiliza como juez y se le proporciona un mensaje del sistema que describe la tarea que queremos que realice el LLM más pequeño y los criterios de evaluación. Un "mensaje del sistema" es una forma de establecer el contexto general y guiar el comportamiento de un LLM. Contextualizado con este mensaje del sistema, el juez LLM puede calificar los resultados del LLM más pequeño, y podemos usar esta puntuación para evaluar qué tan bien le está yendo al LLM más pequeño.

### 2.1: ¡Afina bien, debes hacerlo!

Nuestro modelo adaptado al duende ya es bastante bueno generando respuestas al estilo del duende. Debe ser la suerte de los irlandeses.

Hagamos las cosas más interesantes considerando un estilo diferente, uno que tenga algunos patrones claros pero también mucha variabilidad y espacio para la creatividad. Usaremos el estilo de [Yoda](https://en.wikipedia.org/wiki/Yoda) de Star Wars.

<img src="https://media3.giphy.com/media/v1.Y2lkPTc5MGI3NjExZHcxMGZjZzdwbGV0andseWw3c3h1ODJwOXd 5NHEzbnVtMHk5YWQyayZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/IaWMz9Ln8OWvf66z6k/giphy.webp" />

Su objetivo es intentar ajustar su modelo para generar respuestas al estilo Yoda, usar el juez LLM para evaluar qué tan bien los resultados de su modelo de chat siguen el discurso de Yoda y luego usar esa información para mejorar el modelo.

In [ ]:
# Cargue el conjunto de datos de Yoda-speak y ajuste el modelo usando su función de entrenamiento
train_loader, test_loader = mdl.lab3.create_dataloader(style="yoda")
model = train('''TODO''') # TODO

Comience por definir un mensaje del sistema para el juez LLM, estableciendo el contexto en el que evaluará qué tan bien los resultados de su modelo de chat siguen el discurso de Yoda. ¡Experimente con diferentes indicaciones del sistema para ver cómo afectan la evaluación del juez LLM! Tenga en cuenta que un LLM de mejor juez le brindará una mejor evaluación de qué tan bien le está yendo a su modelo Yoda, y que una mejor evaluación lo ayudará a mejorar su modelo Yoda.

In [ ]:
# ## LLM como juez ###

'''TODO: Experiment with different system prompts to see how they affect the judge LLM's evaluation!
        Come back to this cell after you've generated some text from your model.'''

system_prompt = """
You are an impartial judge that evaluates if text was written by {style}.

An example piece of text from {style} is:
{example}

Now, analyze some new text carefully and respond on if it follows the
same style of {style}. Be critical to identify any issues in the text.
Then convert your feedback into a number between 0 and 10: 10 if the text
is written exactly in the style of {style}, 5 if mixed faithfulness to the
style, or 0 if the text is not at all written in the style of {style}.

Directly answer with the score formatted in a dictionary.
The format of your response should only be the dictionary and nothing else:
{{"score": <score between 0 and 10>}}
"""

style = "Yoda"
example = "The very Republic is threatened, if involved the Sith are. Hard to see, the dark side is. Discover who this assassin is, we must. With this Naboo queen you must stay, Qui-Gon. Protect her. May the Force be with you. A vergence, you say? But you do! Revealed your opinion is. Trained as a Jedi, you request for him? Good, good, young one."

system_prompt = system_prompt.format(style=style, example=example)
print("=== Aviso del sistema ===")
print(system_prompt)

### 2.2: Configuración del juez LLM

En LLM como juez, necesitamos usar un modelo que sea más grande (y por lo tanto más capaz) que nuestro modelo "ejecutivo", en nuestro caso el estilo LFM2 1.2B ajustado. Dado que no es factible cargar modelos más grandes localmente en portátiles, obtendrá experiencia interactuando con estos LLM más grandes a través de una API proporcionada en [OpenRouter](https://openrouter.ai/).

Deberá registrarse en [OpenRouter account](https://openrouter.ai/sign-up) y luego en [generate an API key](https://openrouter.ai/keys). Ejecutar LLM potentes de esta escala cuesta dinero; para los estudiantes del curso presencial, podemos proporcionar un crédito en su cuenta OpenRouter para permitirles ejecutar esta práctica de laboratorio. Acércate al horario de oficina para recibir tu crédito.

A través de la interfaz de OpenRouter, podrá experimentar con diferentes LLM para jueces; aquí sugerimos un posible LLM más grande para comenzar: [Gemini 2.5](https://openrouter.ai/google/gemini-2.5-flash/providers) de Google. Tenga en cuenta que también hay modelos gratuitos disponibles en OpenRouter (por ejemplo, [gemma-2-9b-it:free](https://openrouter.ai/google/gemma-2-9b-it:free)), pero tendrán limitaciones de velocidad si los ejecuta demasiado.

Hemos definido una clase simple, `LLMClient`, para interactuar con la API de OpenRouter. Esta clase tiene un método "preguntar" que recibe un mensaje del usuario y devuelve la respuesta del modelo. Tenga en cuenta que la respuesta del juez LLM estará condicionada al mensaje del sistema que usted proporcione: ¡el mensaje del sistema es fundamental para establecer los criterios para la evaluación!

In [ ]:
OPENROUTER_API_KEY = "" # TODO: agregue su clave API de OpenRouter aquí
assert OPENROUTER_API_KEY != "", "You must set your OpenRouter API key before running this cell!"

model_name = "google/gemini-2.5-flash"
llm = mdl.lab3.LLMClient(model=model_name, api_key=OPENROUTER_API_KEY)

### 2.3: Definición de la métrica de evaluación

¡Genial! Hemos creado nuestro juez LLM, pero aún tenemos que hacerlo cuantitativo. Podemos hacer esto definiendo una métrica que utilice el juez LLM para calificar los resultados del modelo. Hacer esto se simplifica con [Opik library](https://www.comet.com/docs/opik/python-sdk-reference/) de Comet ML, una plataforma para evaluación y evaluación comparativa de LLM.

En laboratorios anteriores, utilizamos Comet para el seguimiento de experimentos, por lo que debe tener una cuenta y una clave API. De lo contrario, puede registrarse para obtener una cuenta de Comet [here](https://www.comet.com/signup?from=llm&utm_source=mit_dl&utm_medium=notebook&utm_campaign=opik) si aún no lo ha hecho. Ahora usaremos la biblioteca Comet Opik para definir una métrica que use el juez LLM para calificar los resultados del modelo.


Opik proporciona un marco para crear métricas de evaluación personalizadas, así como una variedad de métricas prediseñadas para tareas de evaluación comunes. Estas métricas están diseñadas para ayudarlo a medir de manera rápida y efectiva el rendimiento de los resultados de su LLM e incluyen métricas como alucinaciones, relevancia de las respuestas, precisión/recuerdo del contexto y más. Puede obtener más información sobre las métricas disponibles en [`Metrics Overview section`](https://www.comet.com/docs/opik/evaluation/metrics/overview) de la documentación de Opik.

El SDK de Opik Python tiene una clase base para definir métricas, [`base_metric.BaseMetric`](https://www.comet.com/docs/opik/python-sdk-reference/evaluation/metrics/BaseMetric.html). Utilizará esto para definir una métrica personalizada que utilice el juez LLM para evaluar el texto y determinar qué tan bien se adhiere al lenguaje Yoda. Tenga en cuenta que el juez LLM y la métrica se pueden aplicar a cualquier texto, no solo a los resultados del modelo. Es importante tener esto en cuenta, ya que necesitamos tanto un control negativo (texto en el estilo inglés estándar "base") como un control positivo (texto del conjunto de entrenamiento en estilo Yoda-speak) con el cual comparar las generaciones del modelo.

Establezca los criterios de evaluación en el mensaje del sistema y defina la función "puntuación" para evaluar el texto consultando al juez LLM.

In [ ]:
from opik.evaluation.metrics import base_metric, score_result

class LLMJudgeEvaluator(base_metric.BaseMetric):
    def __init__(self, judge: mdl.lab3.LLMClient = None, system_prompt: str = None):
        self.judge = judge
        self.system_prompt = system_prompt
        self.prompt_template = "Evaluate this text: {text}"

    def score(self, text: str, n_tries=20, **kwargs):
        """ Evaluate by asking an LLM to score it. """

        for attempt in range(n_tries):
            try:
                # TODO: convertir el texto a formato de plantilla antes de pasárselo al juez LLM
                prompt = self.prompt_template.format('''TODO''') # TODO

                # TODO: Llame al juez LLM con el mensaje del sistema y la plantilla de mensaje.
                res = self.judge.ask(
                  system='''TODO''',
                  user='''TODO''',
                  max_tokens='''TODO'''
                ) # TODO

                # Extraiga el contenido del asistente de la respuesta de la API
                res = res.choices[0].message.content
                res_dict = json.loads(res)

                max_score = 10 # La puntuación máxima que debe generar el LLM
                score = res_dict["score"] / max_score # Normalizar
                score = max(0.0, min(score, 1.0)) # Recortar entre 0 y 1

                # Devolver el objeto de puntuación
                return score_result.ScoreResult(name="StyleScore", value=score)

            except Exception as e:
                if attempt == n_tries - 1:  # último intento
                    raise e  # Vuelva a generar la excepción si todos los intentos fallaron
                continue  # Inténtalo de nuevo si no es el último intento.

Cree una instancia de su juez Comet Opik utilizando la clase `LLMJudgeEvaluator` y el indicador del sistema.

In [ ]:
judge = LLMJudgeEvaluator(llm, system_prompt=system_prompt)

## 2.4: Evaluación del modelo puntuando con tu juez LLM

Ahora podemos utilizar el juez LLM para calificar los resultados del modelo. Usaremos la `scoring_function` para calificar el texto usando el juez LLM.

Introduzca algunas frases de investigación para comprobar la vibra del juez LLM.

In [ ]:
def scoring_function(text):
    return judge.score(text).value

test_texts = [
    "Tennis is a fun sport. But you must concentrate.",
    "Fun sport, tennis is. But work hard, you must.",
    "Hard to see, the dark side is."
]

for text in test_texts:
    score = scoring_function(text)
    print(f"{text} ==> Score: {score}")

Evaluaremos qué tan bien está funcionando nuestro modelo ajustado calificando los resultados del modelo, así como nuestro texto de estilo base (control negativo) y el texto del conjunto de entrenamiento en estilo Yoda-speak (control positivo).

Genere texto a partir de su modelo haciéndole nuevas preguntas.


In [ ]:
# Genere texto a partir de su modelo haciéndole nuevas preguntas.
def generate_samples_from_test(test_loader, num_samples):
    samples = []
    for test_sample in tqdm(test_loader, total=num_samples):
        test_question = test_sample['instruction'][0]
        with torch.no_grad():
            generated = chat(test_question, only_answer=True, max_new_tokens=100)
        samples.append(generated)
        if len(samples) >= num_samples:
            break
    return samples

n_samples = 20
generated_samples = generate_samples_from_test(test_loader, num_samples=n_samples)

También recopilemos algo de texto de estilo base (`base_samples`) y el texto del conjunto de entrenamiento en estilo Yoda-speak (`style_samples`). Para estos, no necesitaremos generar texto, ya que ya tenemos el texto en el conjunto de datos.

In [ ]:
base_samples = [sample['response'][0] for i, sample in enumerate(train_loader) if i < n_samples]
style_samples = [sample['response_style'][0] for i, sample in enumerate(train_loader) if i < n_samples]

Ahora que tenemos nuestras muestras, podemos calificarlas usando el juez LLM. Usaremos una función de puntuación multiprocesada para calificar las muestras en paralelo, porque cada muestra es independiente y podemos enviarlas todas como solicitudes simultáneas al juez LLM.

In [ ]:
# Cree una función de puntuación multiprocesada para puntuar las muestras en paralelo

os.environ["TOKENIZERS_PARALLELISM"] = "false"
from multiprocessing import Pool

def compute_scores_in_parallel(samples):
    with Pool(processes=10) as pool:
        scores = pool.map(scoring_function, samples)
    return scores

# Calcule e imprima las puntuaciones para el texto de estilo base, el texto generado y el texto del conjunto de entrenamiento en el estilo Yoda-speak.
base_scores = compute_scores_in_parallel(base_samples)
print(f"Base: {np.mean(base_scores):.2f} ± {np.std(base_scores):.2f}")

generated_scores = compute_scores_in_parallel(generated_samples)
print(f"Gen: {np.mean(generated_scores):.2f} ± {np.std(generated_scores):.2f}")

style_scores = compute_scores_in_parallel(style_samples)
print(f"Train: {np.mean(style_scores):.2f} ± {np.std(style_scores):.2f}")

Mire las puntuaciones promedio para cada uno de los tres tipos de texto: ¿qué observa?

También podemos trazar la distribución de puntuaciones para cada uno de los tres tipos de texto.


In [ ]:
import seaborn as sns
import pandas as pd

# Crear un marco de datos limpio
df = pd.DataFrame({
    'Score': [*base_scores, *generated_scores, *style_scores],
    'Type': ['Base']*len(base_scores) + ['Generated']*len(generated_scores) + ['Style']*len(style_scores)
})

# Trama con seaborn
sns.histplot(data=df, x='Score', hue='Type', multiple="dodge", bins=6, shrink=.8)

plt.title('Distribution of Scores')
plt.show()

Utilice estas observaciones para mejorar su modelo. Recuerde que el juez LLM no es perfecto y puede intentar mejorarlo para evaluar mejor los resultados del modelo. Un mejor juez LLM le brindará una mejor evaluación de qué tan bien le está yendo a su modelo Yoda, y esa mejor evaluación lo ayudará a mejorar su modelo Yoda.

## 2.5: Monitoreo con evaluaciones

Así como usamos Opik para evaluar métricas durante el ajuste y las pruebas, también podemos usar Opik para monitorear nuestro LLM una vez que esté implementado. Esto facilita el seguimiento consistente de las mismas métricas tanto en el desarrollo como en la implementación.

En laboratorios anteriores, utilizamos Comet para el seguimiento de experimentos, por lo que debe tener una cuenta y una clave API. De lo contrario, puede registrarse para obtener una cuenta de Comet [here](https://www.comet.com/signup?from=llm&utm_source=mit_dl&utm_medium=notebook&utm_campaign=opik) si aún no lo ha hecho. Configuraremos Opik configurando la clave API y nombrando nuestro proyecto Opik.

In [ ]:
os.environ["OPIK_API_KEY"] = "" # TODO: agregue su clave API OPIK o Comet aquí
assert OPIK_API_KEY != "", "You must set your OPIK or Comet API key before running this cell!"

# Establecer el nombre del proyecto para Opik
os.environ["OPIK_PROJECT_NAME"] = "6S191_Lab3"

opik.configure()

[Tracing](https://www.comet.com/docs/opik/tracing/concepts) lo ayuda a comprender el flujo de un extremo a otro de su solicitud de LLM y a identificar pasos específicos que pueden estar causando problemas.

En el siguiente ejemplo, hacemos una llamada de muestra al chatbot y utilizamos el decorador `@track` de Opik para registrar datos en la interfaz de usuario de Opik, creando un registro de llamadas en vivo a la aplicación. Puede agregar el decorador `@track` a cualquier función para rastrear no solo las llamadas de LLM, sino también otros pasos en el proceso de su aplicación.

In [ ]:
@opik.track
def inference_chat(question, max_new_tokens=32, temperature=0.7, only_answer=False):

    # 1. Construya el mensaje usando la plantilla.
    prompt = template_without_answer.format(question=question)

    # 2. Tokenizar el texto
    input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 3. Analice el modelo para predecir las siguientes probabilidades simbólicas.
    with torch.no_grad():
        outputs = model.generate(**input_ids, do_sample=True, max_new_tokens=max_new_tokens, temperature=temperature)

    # 4. Solo devuelve la respuesta si only_answer es Verdadero
    output_tokens = outputs[0]
    if only_answer:
        output_tokens = output_tokens[input_ids['input_ids'].shape[1]:]

    # 5. Decodifica las fichas
    result = tokenizer.decode(output_tokens, skip_special_tokens=True)

    # Actualizar el seguimiento actual con puntuaciones de evaluación.
    opik_context.update_current_trace(
        feedback_scores=[
            {
                "name": "Yoda style eval",
                "value": scoring_function(result)
            }
        ]
    )

    return result

Ahora puede realizar una llamada de ejemplo a su modelo para ver el seguimiento registrado en Opik. Una vez que ejecute la celda a continuación, debería ver un enlace a su interfaz de usuario de Opik donde se registran sus seguimientos en su proyecto. Siga ese enlace para ver sus rastros en la plataforma Opik.

In [ ]:
# Intentemos charlar con el modelo ahora para ver los trazos producidos con la partitura.
answer = inference_chat(
    "Who was the only non-Jedi to wield a lightsaber in the original Star Wars trilogy?",
    only_answer=True,
    max_new_tokens=32,
)

print(answer)

## 2.6: Conclusión

Experimente tanto con su modelo de chat como con su juez LLM para intentar mejorar la calidad del lenguaje Yoda. El concurso para este laboratorio se basará en los siguientes criterios:
* **Probabilidad de un verdadero lenguaje Yoda según su modelo de chat**: cuanto mejor comprenda su modelo de chat el lenguaje Yoda, estimará una menor pérdida de entropía cruzada para el lenguaje que es un verdadero lenguaje Yoda. Al final de esta práctica de laboratorio, evaluará la probabilidad de obtener una muestra de prueba del verdadero lenguaje Yoda según su modelo de chat. Incluya esta probabilidad en su informe. Esto nos brinda una medida cuantitativa para comparar diferentes modelos de chat (que pueden haber interactuado con diferentes jueces LLM).
* **Experimentos y cambios que intentaste para mejorar tu modelo de chat**: incluye una descripción de los cambios que realizaste y los resultados que observaste.

#### IMPORTANTE: EJECUTE LA SIGUIENTE CELDA PARA IMPRIMIR EL RESULTADO PERO NO MODIFICA SU CONTENIDO.

In [ ]:
# NO CAMBIAR/MODIFICAR ESTA CELDA.
# EJECUTARLO ANTES DE ENVIAR SU ENTRADA AL LABORATORIO.

yoda_test_text = mdl.lab3.yoda_test_text
tokens = tokenizer(yoda_test_text, return_tensors="pt").to(model.device)

# Obtenga la probabilidad logarítmica del modelo.
with torch.no_grad():
    outputs = model(**tokens)
    logits = outputs.logits[:, :-1]
    targets = tokens.input_ids[:, 1:]
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                            targets.reshape(-1))

print(f"Yoda test loglikelihood: {loss.item():.2f}")


# Información de envío

Para participar en el concurso, cargue lo siguiente en el laboratorio [submission site for the Large Language Models Lab](https://www.dropbox.com/request/l2JH7UlrayUl1Ps5ZVZm):

* Cuaderno Jupyter con el código que usaste para generar tus resultados;
* copia del diagrama de barras que muestra las puntuaciones de texto del juez LLM en estilo base, texto generado y texto en verdadero estilo Yoda-speak;
* una descripción escrita de las modificaciones que realizó y de los experimentos que probó;
* una discusión escrita de por qué y cómo estas modificaciones cambiaron el desempeño;
* **el resultado numérico de la última celda de este cuaderno**.

Los envíos sin el resultado de la última celda serán descalificados automáticamente.

**Nombre su archivo en el siguiente formato: `[Nombre]_[Apellido]_LLM`, seguido del formato de archivo (.zip, .ipynb, .pdf, etc.).** Se prefieren los archivos ZIP a los archivos individuales. Si envía archivos individuales, debe nombrar los archivos individuales de acuerdo con la nomenclatura anterior (por ejemplo, `[Nombre]_[Apellido]_LLM_Report.pdf`, etc.).

<img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExdDZsczFmcjcxeWZjbTA2djh5bDN1bzl5 eHJpeHFhdHM0dmczcjkxMyZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/ArrVyXcjSzzxe/giphy.webp" />